# 00 · Deploy & Configure — Azure Cost Analyzer Pipeline

**Idempotent** deployment of the **Azure Cost Analyzer (ACA)** medallion pipeline in the current Fabric workspace.

This notebook recreates `AzureCostAnalyzer_Pipeline` from code so the whole accelerator can be
stood up (or re-synced) in any workspace without hand-wiring the Data Factory canvas.

## What it does
1. Resolves the **current workspace** (from runtime context).
2. Resolves (or creates) the Lakehouse **`AzureCostAnalyzer_LH`**.
3. Discovers the **8 processing notebooks** by name and gets their IDs.
4. Builds the pipeline **DAG + parameters** (exact replica of the deployed definition).
5. **Creates or updates** the Data Pipeline `AzureCostAnalyzer_Pipeline`.
6. *(Optional)* Triggers an on-demand run.

## DAG (what gets built)
```
                                  ┌─ 02_Gold_Calendar
                                  ├─ 03_Gold_FocusMonthly
01_Load_CostManagement_Focus_Data ┼─ 04_Gold_CostSummary ──→ 07_Gold_Anomalies
                                  ├─ 05_Gold_ByTag
                                  ├─ 06_Gold_Reservations
                                  └─ 08_Gold_CostByResource
```
- `01` fans out to the 6 gold builders (all depend only on `01`).
- `07_Gold_Anomalies` depends on `04_Gold_CostSummary` (reads `gold_cost_summary_daily`).

## Prerequisite
Import the 8 notebooks (`01_…` → `08_…`) into this workspace first, keeping the **exact names** —
this notebook looks them up by display name.

## What you still do after this notebook (one-time)
- **Semantic model** — run notebook `09_Build_SemanticModel` to build/refresh the Direct-Lake model `AzureCostAnalyzer_SM` (all tables, relationships, and measures — fully automated).
- *(Optional)* **Data Agent** and the **RayFin app** add-on — see `docs/deployment-guide.md`.
- **Schedule** — set on the pipeline (Pipeline → Schedule) so timing is tunable per environment.

> Safe to re-run. Every step checks first (create-or-update).

In [ ]:
# ============================================================
# PARAMETERS  (override here when deploying to a new workspace)
# ============================================================
LAKEHOUSE_NAME = "AzureCostAnalyzer_LH"
PIPELINE_NAME  = "AzureCostAnalyzer_Pipeline"

# Pipeline-level default parameter values (match the deployed definition).
#   FromMonth / ToMonth are month offsets from "today − 1 day":
#     0 = current month, -1 = previous month, etc.  Load reads [FromMonth .. ToMonth].
FROM_MONTH_DEFAULT    = -1
TO_MONTH_DEFAULT      = 0
RAW_SOURCE_PATH_DEFAULT = "Files/focuscost"

# Optional: trigger an on-demand run after deploy (see §6).
RUN_AFTER_DEPLOY = False

# Activity → notebook mapping + DAG (declarative; edit here to change topology).
#   activity   : pipeline activity name (what shows on the canvas)
#   notebook   : workspace notebook display name to bind
#   depends_on : list of upstream activity names ("Succeeded" condition)
#   params     : pipeline parameters this activity forwards to the notebook
ACTIVITY_SPEC = [
    {"activity": "Load_CostManagement_Focus_Data", "notebook": "01_Load_CostManagement_Focus_Data",
     "depends_on": [],                                "params": ["FromMonth", "ToMonth", "RawSourcePath", "WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_Calendar",                  "notebook": "02_Gold_Calendar",
     "depends_on": ["Load_CostManagement_Focus_Data"], "params": ["WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_FocusMonthly",              "notebook": "03_Gold_FocusMonthly",
     "depends_on": ["Load_CostManagement_Focus_Data"], "params": ["WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_CostSummary",               "notebook": "04_Gold_CostSummary",
     "depends_on": ["Load_CostManagement_Focus_Data"], "params": ["WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_ByTag",                     "notebook": "05_Gold_ByTag",
     "depends_on": ["Load_CostManagement_Focus_Data"], "params": ["WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_Reservations",              "notebook": "06_Gold_Reservations",
     "depends_on": ["Load_CostManagement_Focus_Data"], "params": ["WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_Anomalies",                 "notebook": "07_Gold_Anomalies",
     "depends_on": ["Gold_CostSummary"],            "params": ["WorkspaceId", "LakehouseId"]},
    {"activity": "Gold_CostByResource",            "notebook": "08_Gold_CostByResource",
     "depends_on": ["Load_CostManagement_Focus_Data"], "params": ["WorkspaceId", "LakehouseId"]},
]

# Pipeline parameter data types (used when emitting the per-activity bindings).
PARAM_TYPES = {
    "FromMonth": "int", "ToMonth": "int", "RawSourcePath": "string",
    "WorkspaceId": "string", "LakehouseId": "string",
}

NOTEBOOK_NAMES = [s["notebook"] for s in ACTIVITY_SPEC]
print(f"Pipeline      : {PIPELINE_NAME}")
print(f"Lakehouse     : {LAKEHOUSE_NAME}")
print(f"Activities ({len(ACTIVITY_SPEC)}): {[s['activity'] for s in ACTIVITY_SPEC]}")

In [ ]:
import json
import base64
import time
import requests
import notebookutils

ctx = notebookutils.runtime.context
WORKSPACE_ID = ctx["currentWorkspaceId"]

token   = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
API     = "https://api.fabric.microsoft.com/v1"

print(f"Workspace ID : {WORKSPACE_ID}")

def b64(obj) -> str:
    """Base64-encode a JSON object for a Fabric definition part payload."""
    return base64.b64encode(json.dumps(obj).encode("utf-8")).decode("ascii")

## 1 · Helper functions (REST + long-running operations)

In [ ]:
def list_workspace_items(workspace_id: str, item_type: str | None = None):
    url = f"{API}/workspaces/{workspace_id}/items"
    if item_type:
        url += f"?type={item_type}"
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json().get("value", [])

def get_item_id(workspace_id: str, name: str, item_type: str):
    for item in list_workspace_items(workspace_id, item_type):
        if item["displayName"].lower() == name.lower():
            return item["id"]
    return None

def _wait_lro(initial_response):
    """Resolve a Fabric long-running operation (202) and return final JSON (if any)."""
    if initial_response.status_code in (200, 201):
        return initial_response.json() if initial_response.text else {}
    if initial_response.status_code != 202:
        initial_response.raise_for_status()
    op_url = initial_response.headers.get("Location")
    if not op_url:
        return {}
    for _ in range(90):
        time.sleep(2)
        rr = requests.get(op_url, headers=HEADERS, timeout=30)
        if rr.status_code == 200:
            status = rr.json().get("status", "").lower()
            if status in ("succeeded", "completed"):
                result_url = rr.headers.get("Location")
                if result_url:
                    rd = requests.get(result_url, headers=HEADERS, timeout=60)
                    if rd.status_code == 200 and rd.text:
                        return rd.json()
                return {}
            if status == "failed":
                raise RuntimeError(f"LRO failed: {rr.text}")
    raise RuntimeError("LRO timeout")

## 2 · Resolve (or create) the Lakehouse `AzureCostAnalyzer_LH`

In [ ]:
def get_lakehouse_id(workspace_id: str, name: str):
    r = requests.get(f"{API}/workspaces/{workspace_id}/lakehouses", headers=HEADERS, timeout=60)
    r.raise_for_status()
    for lh in r.json().get("value", []):
        if lh["displayName"].lower() == name.lower():
            return lh["id"]
    return None

lakehouse_id = get_lakehouse_id(WORKSPACE_ID, LAKEHOUSE_NAME)

if lakehouse_id:
    print(f"✓ Lakehouse '{LAKEHOUSE_NAME}' already exists → {lakehouse_id}")
else:
    body = {"displayName": LAKEHOUSE_NAME, "description": "Azure Cost Analyzer — silver/gold storage"}
    r = requests.post(f"{API}/workspaces/{WORKSPACE_ID}/lakehouses",
                      headers=HEADERS, data=json.dumps(body), timeout=120)
    r.raise_for_status()
    lakehouse_id = r.json()["id"]
    print(f"✓ Created lakehouse '{LAKEHOUSE_NAME}' → {lakehouse_id}")

## 3 · Resolve the 7 processing notebooks by name

They must already exist in this workspace (imported with the exact names).

In [ ]:
notebooks   = list_workspace_items(WORKSPACE_ID, "Notebook")
name_to_id  = {nb["displayName"]: nb["id"] for nb in notebooks}

missing = [n for n in NOTEBOOK_NAMES if n not in name_to_id]
if missing:
    raise RuntimeError(
        "Missing notebook(s) in this workspace: " + ", ".join(missing)
        + ". Import them first, keeping the exact display names."
    )

notebook_ids = {n: name_to_id[n] for n in NOTEBOOK_NAMES}
for n, i in notebook_ids.items():
    print(f"  ✓ {n}  →  {i}")

## 4 · Build the pipeline definition (DAG + parameters)

Replicates the deployed `AzureCostAnalyzer_Pipeline`: each `TridentNotebook` activity
forwards the pipeline parameters it needs via `@pipeline().parameters.<name>` expressions.

In [ ]:
def pipeline_param_ref(name: str):
    """Bind a notebook parameter to the pipeline parameter of the same name."""
    return {
        "value": {"value": f"@pipeline().parameters.{name}", "type": "Expression"},
        "type": PARAM_TYPES[name],
    }

def notebook_activity(spec: dict, notebook_id: str):
    return {
        "name": spec["activity"],
        "type": "TridentNotebook",
        "dependsOn": [
            {"activity": d, "dependencyConditions": ["Succeeded"]}
            for d in spec["depends_on"]
        ],
        "policy": {
            "timeout": "0.12:00:00",
            "retry": 0,
            "retryIntervalInSeconds": 30,
            "secureOutput": False,
            "secureInput": False,
        },
        "typeProperties": {
            "notebookId": notebook_id,
            "workspaceId": WORKSPACE_ID,
            "parameters": {p: pipeline_param_ref(p) for p in spec["params"]},
        },
    }

activities = [notebook_activity(s, notebook_ids[s["notebook"]]) for s in ACTIVITY_SPEC]

pipeline_parameters = {
    "FromMonth":     {"type": "int",    "defaultValue": FROM_MONTH_DEFAULT},
    "ToMonth":       {"type": "int",    "defaultValue": TO_MONTH_DEFAULT},
    "RawSourcePath": {"type": "string", "defaultValue": RAW_SOURCE_PATH_DEFAULT},
    "WorkspaceId":   {"type": "string", "defaultValue": WORKSPACE_ID},
    "LakehouseId":   {"type": "string", "defaultValue": lakehouse_id},
}

pipeline_content = {"properties": {"activities": activities, "parameters": pipeline_parameters}}

# Quick DAG sanity print
print("DAG:")
for s in ACTIVITY_SPEC:
    deps = " , ".join(s["depends_on"]) or "—"
    print(f"  {s['activity']:<32} depends on: {deps}")

## 5 · Create (or update) the Data Pipeline

In [ ]:
definition = {
    "parts": [{
        "path": "pipeline-content.json",
        "payload": b64(pipeline_content),
        "payloadType": "InlineBase64",
    }],
}

pipeline_id = get_item_id(WORKSPACE_ID, PIPELINE_NAME, "DataPipeline")

if pipeline_id:
    url = f"{API}/workspaces/{WORKSPACE_ID}/dataPipelines/{pipeline_id}/updateDefinition"
    r = requests.post(url, headers=HEADERS, data=json.dumps({"definition": definition}), timeout=180)
    _wait_lro(r)
    print(f"✓ Pipeline '{PIPELINE_NAME}' updated → {pipeline_id}")
else:
    body = {
        "displayName": PIPELINE_NAME,
        "description": "ACA — daily FOCUS load + gold build (calendar, focus-monthly, cost summary, by-tag, reservations, anomalies)",
        "definition": definition,
    }
    url = f"{API}/workspaces/{WORKSPACE_ID}/dataPipelines"
    r = requests.post(url, headers=HEADERS, data=json.dumps(body), timeout=180)
    result = _wait_lro(r)
    pipeline_id = (result or {}).get("id") or get_item_id(WORKSPACE_ID, PIPELINE_NAME, "DataPipeline")
    print(f"✓ Pipeline '{PIPELINE_NAME}' created → {pipeline_id}")

## 6 · (Optional) Trigger an on-demand run

Set `RUN_AFTER_DEPLOY = True` in the parameters cell to kick off a run right after deploy.
Leave it `False` to deploy only and run via schedule / the FCA pipeline.

In [ ]:
if RUN_AFTER_DEPLOY:
    url = f"{API}/workspaces/{WORKSPACE_ID}/items/{pipeline_id}/jobs/instances?jobType=Pipeline"
    r = requests.post(url, headers=HEADERS, data=json.dumps({}), timeout=120)
    if r.status_code in (200, 201, 202):
        run_url = r.headers.get("Location", "(check Monitoring hub)")
        print(f"✓ On-demand run started. Track it at:\n  {run_url}")
    else:
        raise RuntimeError(f"Run trigger failed: {r.status_code} · {r.text}")
else:
    print("RUN_AFTER_DEPLOY = False → pipeline deployed but not started.")

## ✅ Done — summary

| Resource | Status |
|---|---|
| Lakehouse `AzureCostAnalyzer_LH` | resolved / created |
| Notebooks `01`–`07` | discovered |
| Pipeline `AzureCostAnalyzer_Pipeline` | created / updated (7 activities) |

### Next steps (manual, one-time)
1. **Schedule** the pipeline: open `AzureCostAnalyzer_Pipeline` → **Schedule** → daily (e.g. 06:00 local),
   or chain it after the FCA pipeline with an *Invoke pipeline* activity.
2. **First run**: trigger once (set `RUN_AFTER_DEPLOY = True`, or click **Run**) and confirm green ticks.
3. **Semantic model**: run notebook `09_Build_SemanticModel` to build/refresh `AzureCostAnalyzer_SM`
   (binds the `gold_*` tables, relationships, and measures automatically). `dim_month` drives monthly facts, `dim_date` daily.
4. *(Optional)* **Data Agent** (`AzureCostAnalyzer_Agent`) and the **RayFin app** add-on for a conversational + web UI — see `docs/deployment-guide.md`.

> To deploy in a **new** workspace: import the 00–09 notebooks there, open this notebook in that workspace,
> adjust the PARAMETERS cell if names differ, and run top-to-bottom. The pipeline `LakehouseId` /
> `WorkspaceId` defaults are resolved from the live workspace, so notebooks receive the right IDs at runtime.